# LexAI One-Click Databricks Runner

Run this notebook to initialize the notebook-06 engine and optionally expose API/UI from the same cluster session.

## What this notebook does
- Validates paths and cluster context
- Installs missing application dependencies (optional)
- Initializes notebook 06 runtime via adapter
- Runs smoke-test legal queries
- Optionally starts FastAPI and Streamlit

## Prerequisite
- `04_generate_embedding_Test.ipynb` and `06_High-precision_QA_Legal_Reasoning_Engine.ipynb` logic should be available in this repo.
- Embedding Delta data should exist in the configured Volume/table.


In [0]:
# CELL 1: Runtime Flags (edit if needed)
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = True

# Smoke test queries
SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Server ports
FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

# Optional: override if repo location differs
REPO_DIR_OVERRIDE = ""

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': True, 'FASTAPI_PORT': 8000, 'STREAMLIT_PORT': 8501}


In [0]:
# CELL 2: Resolve repo path and set working directory
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists():
            nb = p / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"
            if nb.exists():
                return p
            raise FileNotFoundError(f"REPO_DIR_OVERRIDE exists but notebook 06 not found at: {nb}")

    cwd = Path(os.getcwd())
    if (cwd / "apps" / "fastapi_app.py").exists() and (cwd / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb").exists():
        return cwd

    # Databricks paths: prefer roots with both apps and notebook 06 present.
    candidates = [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]
    for root in candidates:
        if not root.exists():
            continue
        for app_file in root.rglob("apps/fastapi_app.py"):
            repo_dir = app_file.parent.parent
            nb = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"
            if nb.exists():
                return repo_dir

    raise FileNotFoundError("Could not locate repo root containing both apps/fastapi_app.py and notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb")


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
log(f"Working directory: {Path.cwd()}")
print("[CELL 2] OK")


---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-8297215751238367>, line 39
     34                 return repo_dir
     36     raise FileNotFoundError("Could not locate repo root containing both apps/fastapi_app.py and notebooks/06_High-precision_QA_Legal_Reasoning_Engine.ipynb")
---> 39 REPO_DIR = resolve_repo_dir()
     40 os.chdir(REPO_DIR)
     41 if str(REPO_DIR) not in sys.path:

File <command-8297215751238367>, line 30, in resolve_repo_dir()
     28 if not root.exists():
     29     continue
---> 30 for app_file in root.rglob("apps/fastapi_app.py"):
     31     repo_dir = app_file.parent.parent
     32     nb = repo_dir / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"

File /usr/lib/python3.10/pathlib.py:1047, in Path.rglob(self, pattern)
   1045     raise NotImplementedError("Non-relative patterns are unsupported")
   1046 selector = _make_sel

In [0]:
# CELL 3: Optional dependency install (idempotent)
import importlib.metadata as ilm
import subprocess

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

# App + notebook-06 runtime dependencies
required_pkgs = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
]
missing = []
for pkg in required_pkgs:
    try:
        ilm.version(pkg)
    except Exception:
        missing.append(pkg)

print("[CELL 3] Missing packages:", missing)
if missing and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)] + missing
    print("[CELL 3] Installing missing packages...")
    subprocess.check_call(cmd)
    print("[CELL 3] Installation completed")
elif missing and not AUTO_INSTALL_MISSING:
    raise RuntimeError(f"Missing required packages: {missing}. Set AUTO_INSTALL_MISSING=True.")
else:
    print("[CELL 3] All required packages already installed")


In [0]:
# CELL 4: Validate Spark and Databricks context
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"

try:
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
except Exception:
    pass

try:
    org_id = spark.conf.get("spark.databricks.clusterUsageTags.orgId")
except Exception:
    pass

try:
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    pass

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter
from pathlib import Path
import importlib
import tempfile
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

nb_candidates = [
    Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb",
    Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb",
]

# Search filesystem workspace paths if neither candidate exists
if not any(p.exists() for p in nb_candidates):
    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hits = list(root.rglob("06_High-precision_QA_Legal_Reasoning_Engine.ipynb"))
        if hits:
            nb_candidates.append(hits[0])
            break

NOTEBOOK_06_PATH = None
for cand in nb_candidates:
    if cand.exists():
        NOTEBOOK_06_PATH = cand
        break

# Databricks Workspace export fallback (for workspace notebooks not present as local FS files)
def _workspace_export_candidates():
    out = []
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        cur = ctx.notebookPath().get()  # e.g. /Users/<email>/.../notebooks/07_one_click_lexai_runner
        if cur:
            cur_path = Path(cur)
            # sibling notebook 06 in same folder
            out.append(str(cur_path.parent / "06_High-precision_QA_Legal_Reasoning_Engine"))
            out.append(str(cur_path.parent / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"))
            # repo root + notebooks folder
            repo_root_guess = cur_path.parent.parent
            out.append(str(repo_root_guess / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine"))
            out.append(str(repo_root_guess / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"))
            # with /Workspace prefix variants
            for x in list(out):
                if not x.startswith("/Workspace"):
                    out.append("/Workspace" + x)
    except Exception:
        pass

    # Deduplicate preserving order
    dedup = []
    seen = set()
    for p in out:
        if p not in seen:
            seen.add(p)
            dedup.append(p)
    return dedup


if NOTEBOOK_06_PATH is None:
    exported_tmp = None
    ws_candidates = _workspace_export_candidates()
    print("[CELL 5] Workspace export candidates:")
    for w in ws_candidates:
        print(" -", w)

    for ws_path in ws_candidates:
        try:
            # JUPYTER format gives full notebook JSON string
            payload = dbutils.workspace.export(ws_path, format="JUPYTER")
            if payload and str(payload).strip():
                tmp_dir = Path(tempfile.gettempdir())
                exported_tmp = tmp_dir / "lexai06_exported.ipynb"
                exported_tmp.write_text(str(payload), encoding="utf-8")
                NOTEBOOK_06_PATH = exported_tmp
                print(f"[CELL 5] Exported notebook 06 from workspace path: {ws_path}")
                break
        except Exception as _e:
            continue

print("[CELL 5] Candidate file paths:")
for c in nb_candidates:
    print(" -", c, "exists=", c.exists())

if NOTEBOOK_06_PATH is None:
    raise FileNotFoundError(
        "Notebook 06 not found in filesystem or workspace export fallback. "
        "Pull latest repo and ensure notebook 06 exists in Databricks workspace."
    )

print("[CELL 5] Using notebook path:", NOTEBOOK_06_PATH)

try:
    engine = NotebookEngine(notebook_path=NOTEBOOK_06_PATH)
    status = engine.initialize()
except Exception as e:
    print("[CELL 5] Initialization failed:", e)
    import traceback
    traceback.print_exc()
    raise

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")


In [0]:
# CELL 6: Smoke test queries (citation-rich output check)
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


In [0]:
# CELL 7: Start FastAPI server (background thread)
import threading
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")

if START_FASTAPI:
    if FASTAPI_THREAD is not None and FASTAPI_THREAD.is_alive():
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")
    else:
        from apps.fastapi_app import app

        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI started on 0.0.0.0:{FASTAPI_PORT}")

    print("[CELL 7] Local health URL:", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    try:
        if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
            proxy_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{FASTAPI_PORT}/health"
            print("[CELL 7] Driver proxy URL:", proxy_url)
    except Exception:
        pass
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


In [0]:
# CELL 8: Optional FastAPI smoke call
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] /v1/legal/answer status:", r.status_code)
        print("[CELL 8] answer preview:", r.json().get("answer", "")[:500])
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


In [0]:
# CELL 9: Optional Streamlit start (blocking cell)
import os
import subprocess

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port",
        str(STREAMLIT_PORT),
        "--server.address",
        "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        ui_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{STREAMLIT_PORT}/"
        print("[CELL 9] Streamlit URL:", ui_url)
    print("[CELL 9] This cell is blocking while Streamlit is running.")
    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


In [0]:
# CELL 10: Stop helper (run when needed)
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")
